# 06 — Bring your Case information: review first, solve second

**Goal:** start from the information you actually have, make missing/assumed values visible, and only then run a bounded solver-backed example.

This lesson is intentionally different from a long CLI tour. Think of it as a small engineering workspace:

1. **Check the runtime** — CEPT + OpenDSS must be healthy.
2. **Paste Case information** — notes, values, or a summary of files you have.
3. **Review gaps** — choose `strict`, `assisted`, or `exploratory`; nothing is guessed silently.
4. **Optional OpenCode help** — let the agent organize the intake, not manufacture evidence.
5. **Run one bounded demo** — inspect the exact solver receipt.

> **Claim boundary:** `WORKFLOW_VALIDATED` only. A passing notebook is not project validation, field validation, or PowerFactory agreement.


In [1]:
# @title Setup — run once, then read the results below
import urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
exec(compile(_blob, "lesson helper", "exec"))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
lesson helpers ready: cli/read/table/cards + WORKSPACE.


## Step 0 — Is the solver path healthy?

Run this once. The cards keep the important state visible without dumping pages of JSON.


In [2]:
capabilities = cli('capability', 'show')
doctor = cli('system', 'doctor')
cards([
    ('Runtime', doctor['status'], 'trial solve must pass'),
    ('Engine', doctor['engine'], 'solver identity from CEPT'),
    ('Studies', len(capabilities['capabilities']), 'public candidate study types'),
], title='0 · Runtime status')
assert doctor['status'] == 'PASS'
assert doctor['engine'] == 'opendss'
print(f"Runtime: {doctor['status']} | engine={doctor['engine']} | public capabilities={len(capabilities['capabilities'])}")


$ cept capability show


→ exit 0


$ cept system doctor


→ exit 0


Runtime: PASS | engine=opendss | public capabilities=4


## Step 1 — Paste the Case information you have

Edit only the box below. It may be incomplete. In a real project the same idea can start from several local files (notes, CSV/Excel, JSON/YAML, manuals, figures, or model files). The full CEPT/OpenCode front desk uses deterministic readers first and normalizes the rest into the canonical Case intake.

Choose a review policy:

| Policy | What it allows |
| --- | --- |
| `strict` | source/derived values only |
| `assisted` | may use a **named documented default** after your explicit approval |
| `exploratory` | may also use an **AI-selected assumption** after your explicit approval; demonstrator only |

Every policy blocks unresolved required inputs. Research work stays source-backed.


In [3]:
POLICY = 'assisted'  # strict | assisted | exploratory
CASE_INFO = '''
11 kV feeder
Transformer rating: 5 MVA
Load: approximately 3 MW
PV: 1 MW
Transformer impedance: not available yet
'''.strip()

Path('case-info.txt').write_text(CASE_INFO + '\n', encoding='utf-8')
cards([
    ('Policy', POLICY, 'you control fallback permission'),
    ('Input', 'case-info.txt', 'original teaching note is preserved'),
], title='1 · Starting information')
print(f'Saved case-info.txt | policy={POLICY}')


Saved case-info.txt | policy=assisted


## Step 2 — Review what is known and what is missing

The next cell creates a **review ledger**, not a solver Case. Notice that the missing transformer impedance stays `unresolved`; no typical value is inserted silently.


In [4]:
resolution_payload = {
    'schema': 'cept-case-info-resolution-v1',
    'policy': POLICY,
    'items': [
        {'field': 'network.nominal_kv', 'status': 'source', 'value': 11.0, 'source_ref': 'case-info.txt'},
        {'field': 'transformer.rating_mva', 'status': 'source', 'value': 5.0, 'source_ref': 'case-info.txt'},
        {'field': 'load.p_mw', 'status': 'source', 'value': 3.0, 'source_ref': 'case-info.txt'},
        {'field': 'pv.p_mw', 'status': 'source', 'value': 1.0, 'source_ref': 'case-info.txt'},
        {'field': 'transformer.impedance_pu', 'status': 'unresolved'},
    ],
    'notes': ['Teaching intake only; no user Case has been solved.'],
}
Path('case-info-resolution.json').write_text(json.dumps(resolution_payload, indent=2) + '\n', encoding='utf-8')

schema_status = 'available'
try:
    from cept.schema import CaseInfoResolutionLedger
    ledger = CaseInfoResolutionLedger.model_validate(resolution_payload)
    blockers = ledger.blockers(research=False)
except ImportError:
    # Older pinned public wheels can still display the lesson before the next release.
    schema_status = 'upgrade wheel for typed validation'
    blockers = [item for item in resolution_payload['items'] if item['status'] == 'unresolved']

cards([
    ('Known', len(resolution_payload['items']) - len(blockers), 'source/derived or approved values'),
    ('Unresolved', len(blockers), 'must be resolved before a user-case solve'),
    ('Typed ledger', schema_status, 'CEPT intake contract'),
], title='2 · Case-information review')
print(f'Case-info review: BLOCKED | unresolved={len(blockers)} | next=obtain/approve a resolution before a user-case solve')


Case-info review: BLOCKED | unresolved=1 | next=obtain/approve a resolution before a user-case solve


### What should happen next?

For `transformer.impedance_pu`, a good assistant should first suggest where to get the value: transformer nameplate/test report, manufacturer datasheet, utility model, or the original network model.

If you deliberately want a demonstrator instead:

- in **assisted** mode, only choose a documented default that has an identifiable source/version and explicitly approve it;
- in **exploratory** mode, you may explicitly let AI choose a placeholder, but the ledger must say `ai_selected_assumption`, record the reason, and keep `approved_by_user: true`;
- neither choice becomes measured/project evidence.


## Step 3 — Optional: let OpenCode organize the intake

This step is optional. Add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to **Colab Secrets**, enable notebook access, then rerun the cell. The key is read into memory and is never printed or written to the workspace.

The agent is instructed to review `case-info.txt` and the ledger, explain missing data, and stop before a solver run while required inputs are unresolved.


In [5]:
AGENT_COMMAND = r'''---
description: Review Case information without inventing engineering evidence
---
Work only in the current teaching workspace. Review the learner objective plus case-info.txt and case-info-resolution.json. Accept mixed local source formats as intake evidence, but never pretend every format has a deterministic CEPT parser. Use source, derived, unresolved, documented_default, and ai_selected_assumption exactly. Never infer approval from silence. strict allows source/derived only; assisted may use an explicitly approved named default; exploratory may also use an explicitly approved AI-selected demonstrator assumption. If anything required remains unresolved, explain what is missing, why it matters, where to obtain it, and stop before the solver. CEPT Public uses noun+verb CLI; never invent commands. WORKFLOW_VALIDATED is not PROJECT_VALIDATED.

$ARGUMENTS
'''

def _colab_secret():
    names = ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY')
    for name in names:
        if os.environ.get(name, '').strip():
            return name
    try:
        from google.colab import userdata
    except ImportError:
        return ''
    for name in names:
        try:
            value = (userdata.get(name) or '').strip()
        except Exception:
            continue
        if value:
            os.environ[name] = value
            return name
    return ''

def review_with_opencode():
    in_colab = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
    key_name = _colab_secret() if in_colab else ''
    if not in_colab or not key_name:
        print('Optional agent: skipped outside interactive Colab or without a configured provider key.')
        return None
    opencode = Path.home() / '.opencode' / 'bin' / 'opencode'
    if not opencode.is_file():
        subprocess.run('curl -fsSL https://opencode.ai/install | bash', shell=True, check=True)
    command_dir = Path.cwd() / '.opencode' / 'commands'
    command_dir.mkdir(parents=True, exist_ok=True)
    (command_dir / 'cept.md').write_text(AGENT_COMMAND, encoding='utf-8')
    prompt = f'Review my Case information using policy {POLICY}. Do not run a user-case solver while required inputs are unresolved.'
    completed = subprocess.run([str(opencode), 'run', '--command', 'cept', prompt], text=True, capture_output=True, check=False)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or 'OpenCode review failed')
    return completed.stdout

AGENT_REVIEW = review_with_opencode()


Optional agent: skipped outside interactive Colab or without a configured provider key.


## Step 4 — Run one bounded solver-backed demonstration

The sample Case information above is intentionally incomplete, so **we do not solve it**. Instead, this final cell runs the bundled IEEE13 load-flow demo and verifies the persisted artifacts. This keeps the lesson honest: the numerical result comes from OpenDSS, while the incomplete learner Case remains blocked.


In [6]:
run_dir = Path.cwd() / 'runs' / '06-bounded-load-flow'
summary = cli('study', 'demo', 'load-flow', '--network', 'ieee13', '--out', run_dir, '--force')
verification = cli('study', 'verify', run_dir)
assert verification['passed'] is True
assert verification['claim'] == 'WORKFLOW_VALIDATED'
cards([
    ('Solver run', verification['status'], 'persisted run verified'),
    ('Claim', verification['claim'], 'workflow evidence only'),
    ('Study', verification['study_type'], 'bundled IEEE13 demo'),
    ('Engine', verification['engine'], 'solver identity from receipt'),
], title='4 · Solver evidence')
print(f"Demo: {verification['status']} | claim={verification['claim']} | study={verification['study_type']} | engine={verification['engine']}")


$ cept study demo load-flow --network ieee13 --out '<notebook-workspace>\runs\06-bounded-load-flow' --force


→ exit 0


$ cept study verify '<notebook-workspace>\runs\06-bounded-load-flow'


→ exit 0


Demo: PASS | claim=WORKFLOW_VALIDATED | study=load_flow | engine=opendss


## What to take away

- **Flexible input does not mean flexible truth.** OpenCode may organize messy information, but everything must converge on one canonical typed Case.
- **Missing stays missing** until you provide evidence or explicitly approve an allowed demonstrator fallback.
- **Default and AI-selected are different.** A documented default is named/versioned; an AI-selected assumption is model judgement. Both must stay visible.
- **Solver truth stays separate.** The final PASS above belongs to the bundled IEEE13 demo, not to the incomplete Case information in this notebook.

### Try it

Replace `CASE_INFO` with your own short description. Keep one important field missing and see whether your review remains blocked. Then add the real value from a source and update its ledger status to `source`.
